In [4]:
import torch
import torch.nn as nn

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.d = d
        self.beta = nn.Parameter(torch.zeros(d))
        self.gamma = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.float()
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        return (
            self.beta
            + self.gamma * (x - mean) / torch.sqrt(std * std + self.eps)
        ).to(in_dtype)


class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.d = d
        self.gamma = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.float()
        rms = (x * x).mean(dim=-1, keepdim=True)
        self.activations["rms"] = rms
        return (self.gamma * x / torch.sqrt(rms + self.eps)).to(in_dtype)

In [ ]:
class RMSNormFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, gamma, eps):
        in_dtype = x.dtype
        x32 = x.float()
        rms = torch.sqrt((x32 * x32).mean(dim=-1, keepdim=True) + eps)
        ctx.save_for_backward(x, gamma, rms)
        return (gamma * x32 / rms).to(in_dtype)

    @staticmethod
    def backward(ctx, grad_out):
        d = grad_out.shape[-1]
        x, gamma, rms = ctx.saved_tensors
        x32 = x.float()
        gamma32 = gamma.float()
        r = 1 / rms
        grad_out32 = grad_out.float()
        u =  grad_out32 * gamma32
        grad_x = r * u - (r ** 3 / d) * ((u * x32).sum(dim=-1, keepdim=True)) * x32
        grad_gamma = (grad_out32 * x32 * r).reshape(-1, x32.shape[-1]).sum(dim=0)
        return grad_x.to(x.dtype), grad_gamma.to(gamma.dtype), None


In [ ]:
import torch.nn.functional as F


class MHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, pad_mask=None):
        B, S, d = x.shape
        Q, K, V = self.W(x).chunk(3, dim=-1)

        Q = Q.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        K = K.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V = V.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)

        attention_scores = Q @ K.transpose(-1, -2) / self.d_head**0.5

        allowed = torch.tril(
            torch.ones(S, S, dtype=torch.bool, device=x.device)
        )
        allowed = allowed[None, None]  
        if pad_mask is not None:
            allowed = allowed & pad_mask[:, None, None, :]  

        attention_scores = attention_scores.masked_fill(
            ~allowed, torch.finfo(attention_scores.dtype).min
        )
        attention_weights = F.softmax(attention_scores, dim=-1, dtype=torch.float32)
        attention_weights = attention_weights * allowed.any(
            dim=-1, keepdim=True
        )  # empty rows -> 0
        attention_weights = attention_weights.to(V.dtype)

        A = attention_weights @ V
        A = A.transpose(1, 2).contiguous().reshape(B, S, d)

        return self.W_o(A)

In [ ]:
class MHAFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, W, W_o, num_heads):
        B, S, d = x.shape
        d_head = d // num_heads
        Q, K, V = (x @ W).chunk(3, dim=-1)

        Q = Q.reshape(B, S, num_heads, d_head).transpose(1, 2)
        K = K.reshape(B, S, num_heads, d_head).transpose(1, 2)
        V = V.reshape(B, S, num_heads, d_head).transpose(1, 2)
        ctx.num_heads = num_heads

        scores = Q @ K.transpose(-1, -2) / d_head ** 0.5
        mask = torch.tril(torch.ones(S, S, dtype=torch.bool, device=x.device))

        scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores.float(), dim=-1)
        weights = weights.to(V.dtype)

        A = weights @ V
        A = A.transpose(1, 2).contiguous().reshape(B, S, d)
        
        ctx.save_for_backward(x, W,  Q, K, V, weights, A, W_o)
        return A @ W_o

        

    @staticmethod
    def backward(ctx, grad_out):
        B, S, d = grad_out.shape
        num_heads = ctx.num_heads
        d_head = d // num_heads
        x, W, Q, K, V, weights, A, W_o = ctx.saved_tensors
        grad_A = grad_out @ W_o.T
        grad_A = grad_A.reshape(B, S, num_heads, d_head).transpose(1, 2)
        grad_W_o = (A.transpose(-1, -2) @ grad_out).sum(dim=0)
        grad_weights = grad_A @ V.transpose(-1, -2)
        grad_V = weights.transpose(-1, -2) @ grad_A
        grad_scores = weights * (grad_weights - (weights * grad_weights).sum(dim=-1, keepdim=True))
        grad_Q = grad_scores @ K / d_head ** 0.5
        grad_K = grad_scores.transpose(-1, -2) @ Q / d_head ** 0.5

        grad_Q = grad_Q.transpose(1, 2).contiguous().reshape(B, S, d)
        grad_K = grad_K.transpose(1, 2).contiguous().reshape(B, S, d)
        grad_V = grad_V.transpose(1, 2).contiguous().reshape(B, S, d)

        M = torch.cat((grad_Q, grad_K, grad_V), dim=-1)

        grad_W = (x.transpose(-1, -2) @ M).sum(dim=0)
        grad_x = M @ W.T

        return (grad_x, grad_W, grad_W_o, None)




In [ ]:
class GQA(nn.Module):
    def __init__(self, d_model, num_kv_heads, num_q_heads):
        super().__init__()
        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.d_head = d_model // num_q_heads
        self.W = nn.Linear(
            d_model, d_model + 2 * num_kv_heads * self.d_head, bias=False
        )
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, d = x.shape
        g = self.num_q_heads // self.num_kv_heads
        Q, K, V = self.W(x).split([d, d // g, d // g], dim=-1)

        Q = (
            Q.reshape(B, S, self.num_q_heads, self.d_head)
            .reshape(B, S, g, self.num_kv_heads, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )
        K = (
            K.reshape(B, S, self.num_kv_heads, self.d_head)
            .reshape(B, S, 1, self.num_kv_heads, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )
        V = (
            V.reshape(B, S, self.num_kv_heads, self.d_head)
            .reshape(B, S, 1, self.num_kv_heads, self.d_head)
            .permute(0, 2, 3, 1, 4)
        )

        scores = Q @ K.transpose(-1, -2) / self.d_head**0.5

        mask = torch.tril(torch.ones(S, S, dtype=torch.bool, device=x.device))
        masked_attention_scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(masked_attention_scores, dim=-1)

        A = weights @ V

        A = (
            A.permute(0, 3, 1, 2, 4)
            .contiguous()
            .reshape(B, S, self.num_q_heads, self.d_head)
            .reshape(B, S, d)
        )

        return self.W_o(A)

In [ ]:
class MHAKV(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, kv_cache=None):
        B, S, d = x.shape
        Q, K, V = self.W(x).chunk(3, dim=-1)

        if kv_cache is not None:
            old_K, old_V = kv_cache
            K = torch.concat((old_K, K), dim=1)
            V = torch.concat((old_V, V), dim=1)

        new_kv_cache = (K, V)
        _, S_kv, _ = K.shape

        Q = Q.reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        K = K.reshape(B, S_kv, self.n_heads, self.d_head).transpose(1, 2)
        V = V.reshape(B, S_kv, self.n_heads, self.d_head).transpose(1, 2)

        attention_scores = Q @ K.transpose(-1, -2) / self.d_head**0.5

        mask = torch.tril(
            torch.ones(S, S_kv, dtype=torch.bool, device=x.device),
            diagonal=S_kv - S,
        )
        masked_attention_scores = attention_scores.masked_fill(
            ~mask, float("-inf")
        )

        attention_weights = F.softmax(masked_attention_scores, dim=-1)

        A = attention_weights @ V
        A = A.transpose(1, 2).contiguous().reshape(B, S, d)

        return self.W_o(A), new_kv_cache

In [ ]:
class RoPE(nn.Module):
    def __init__(self, d_model, base, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.base = base
        self.max_seq_len = max_seq_len

        pos = torch.arange(0, max_seq_len).unsqueeze(dim=-1)
        inv_freq = (1 / base) ** (torch.arange(0, d_model, 2) / d_model)
        PE = pos * inv_freq
        sin = torch.repeat_interleave(torch.sin(PE), repeats=2, dim=-1)
        cos = torch.repeat_interleave(torch.cos(PE), repeats=2, dim=-1)
        self.register_buffer("sin", sin)
        self.register_buffer("cos", cos)

    def rotate_half(self, x):
        u = x[..., 0::2]
        v = x[..., 1::2]
        return torch.stack((-v, u), dim=-1).flatten(-2, -1)

    def forward(self, Q, K):
        T = Q.shape[-2]
        Q_r = self.rotate_half(Q)
        K_r = self.rotate_half(K)
        Q = self.cos[: T] * Q + self.sin[: T] * Q_r
        K = self.cos[: T] * K + self.sin[: T] * K_r
        return Q, K

In [44]:
def NN(X, C):
    K = C.shape[0]
    L = X.unsqueeze(dim=1) - C
    return torch.argmin((L * L).sum(dim=-1), dim=1)


def NN2(X, C):
    N = X.shape[0]
    result = []
    for i in range(N):
        Y = X[i]
        result.append(torch.argmin(((Y - C) ** 2).sum(dim=-1)).item())
    return torch.tensor(result)


def NN3(X, C):
    C2 = (C * C).sum(dim=-1)
    D = X @ C.T
    L = C2 - 2 * D
    return torch.argmin(L, dim=-1)


def NN_chunked(X, C, B):
    N = X.shape[0]
    C2 = (C * C).sum(dim=-1)
    out = torch.zeros(X.shape[0], dtype=X.dtype)
    for i in range(0, N, B):
        out[i : i + B] = torch.argmin(C2 - 2 * X[i : i + B] @ C.T, dim=-1)
    return out


def NN_chunked_2d(X, C, B1, B2):
    N = X.shape[0]
    K = C.shape[0]
    out = torch.zeros(N, dtype=X.dtype, device=X.device)

    for i in range(0, N, B1):
        best_val = torch.full((B1,), float("inf"))
        best_idx = torch.zeros(B1)
        for j in range(0, K, B2):
            C_chunk = C[j : j + B2]
            C2 = (C_chunk**2).sum(dim=-1)  # [B2,]
            scores = C2 - 2 * X[i : i + B1] @ C_chunk.T #[B1, B2]
            val, idx = scores.min(dim=-1)
            better = val < best_val
            best_val = torch.where(better, val, best_val)
            best_idx = torch.where(better, idx + j, best_idx)
        out[i : i + B1] = best_idx

    return out


X = torch.tensor([[1, 11], [2, 22], [3, 33], [4, 44]])
C = torch.tensor([[1, 10], [3, 35]])
print(NN(X, C))
print(NN2(X, C))
print(NN3(X, C))
print(NN_chunked(X, C, 2))
print(NN_chunked_2d(X, C, 2, 2))

tensor([0, 0, 1, 1])
tensor([0, 0, 1, 1])
tensor([0, 0, 1, 1])
tensor([0, 0, 1, 1])
tensor([0, 0, 1, 1])


In [ ]:
import heapq

def topk_retrieval(Q: torch.Tensor, E: torch.Tensor, k: int) -> torch.Tensor:
    inverse_E = 1 / torch.sqrt((E * E).sum(dim=-1))  # [M,]
    D = Q @ E.T
    scores = D * inverse_E
    _, idx = scores.topk(k=k, dim=-1)
    return idx

def topk_retrieval_chunked(Q: torch.Tensor, E: torch.Tensor, k: int, B: int) -> torch.Tensor:
    M = E.shape[0]
    N = Q.shape[0]
    # best_val = torch.ful((N, k), float("-inf"))
    # best_idx = torch.zeros(N, k)
    vals = [[(float("-inf"), 0) for _ in range(k)] for _ in range(N)]
    for i in range(0, M, B):
        E_chunk = E[i : i + B]
        inverse_E = 1 / torch.sqrt((E_chunk * E_chunk).sum(dim=-1))  # [B,]
        D = Q @ E_chunk.T   #[N, B]
        scores = D * inverse_E
        for val, idx in scores.topk(k=k, dim=-1):
            if val > vals[0][0]:
                heapq.heappop(vals)
                heapq.heappush(vals, (val, idx))
    result = torch.tensor(vals)
    result.sort(dim=-1)
    return result
        

print(
    topk_retrieval(
        torch.tensor([[1, 2], [3, 4], [5, 6]]),
        torch.tensor([[10, 11], [11, 12], [13, 14]]),
        1,
    )
)

print(topk_retrieval_chunked(
        torch.tensor([[1, 2], [3, 4], [5, 6]]),
        torch.tensor([[10, 11], [11, 12], [13, 14]]),
        1, 3
    ))

torch.Size([3, 1])
